# Code tạo Manifest csv SUB789


In [1]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter


def get_last_number(text):
    """
    Lấy cụm số cuối chuỗi.
    Ví dụ:
    Subject7 -> 7
    Activity10 -> 10
    Trial2 -> 2
    Camera12 -> 12
    """
    match = re.search(r"\d+$", str(text))
    if match:
        return match.group()
    return None


def normalize_image_timestamp_strict(filename_stem):
    """
    Nhận timestamp ảnh dạng:
    YYYY-MM-DDTHH_MM_SS
    hoặc
    YYYY-MM-DDTHH_MM_SS.microsecond

    Nếu phía sau có _copy, _1, _2... thì tự cắt bỏ.
    """

    text = str(filename_stem).strip()
    
    pattern = r"^(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2}(?:\.\d+)?)(?:[\s_-].*)?$"

    match = re.match(pattern, text)

    if not match:
        print(filename_stem)
        return None

    # Lấy phần timestamp hợp lệ
    text = match.group(1)

    date_part, time_part = text.split("T", 1)

    # Chỉ thay 2 dấu _ đầu trong phần HH_MM_SS thành :
    time_part = time_part.replace("_", ":", 2)

    return date_part + "T" + time_part

def build_frame_manifest(
    image_root,
    label_csv_path,
    output_csv_path,
    label_col="Label"
):
    image_root = Path(image_root)

    rows = []

    skipped_bad_structure = []
    skipped_bad_number = []
    skipped_non_timestamp_name = []

    suffix_counter = Counter()
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

    total_files_seen = 0
    total_image_files_seen = 0

    for img_path in image_root.rglob("*"):

        if not img_path.is_file():
            continue

        total_files_seen += 1
        suffix_counter[img_path.suffix.lower()] += 1

        if img_path.suffix.lower() not in image_extensions:
            continue

        total_image_files_seen += 1

        parts = img_path.relative_to(image_root).parts

        # Expect:
        # Subject7 / Activity1 / Trial2 / Camera1 / timestamp.png
        if len(parts) != 5:
            skipped_bad_structure.append({
                "image_path": str(img_path),
                "relative_path": str(img_path.relative_to(image_root)),
                "parts": str(parts),
                "num_parts": len(parts)
            })
            continue

        subject_raw, activity_raw, trial_raw, camera_raw, filename = parts

        subject = get_last_number(subject_raw)
        activity = get_last_number(activity_raw)
        trial = get_last_number(trial_raw)
        camera = get_last_number(camera_raw)

        if None in [subject, activity, trial, camera]:
            skipped_bad_number.append({
                "image_path": str(img_path.relative_to(image_root)),
                "Subject_raw": subject_raw,
                "Activity_raw": activity_raw,
                "Trial_raw": trial_raw,
                "Camera_raw": camera_raw,
                "Subject": subject,
                "Activity": activity,
                "Trial": trial,
                "Camera": camera
            })
            continue

        timestamp = normalize_image_timestamp_strict(img_path.stem)

        if timestamp is None:
            skipped_non_timestamp_name.append({
                "image_path": str(img_path),
                "filename_stem": img_path.stem,
                "reason": "Tên file không phải timestamp gốc, có thể là copy/_1/_2 hoặc format khác"
            })
            continue

        rows.append({
            "image_path": str(img_path.relative_to(image_root)),
            "Subject": str(subject),
            "Activity": str(activity),
            "Trial": str(trial),
            "Camera": str(camera),
            "Timestamp": timestamp
        })

    image_df = pd.DataFrame(rows)

    label_df = pd.read_csv(label_csv_path)

    key_cols = ["Timestamp", "Subject", "Activity", "Trial"]

    missing_cols = [col for col in key_cols if col not in label_df.columns]
    if missing_cols:
        raise ValueError(f"CSV thiếu các cột bắt buộc: {missing_cols}")

    if label_col not in label_df.columns:
        print(f"Cảnh báo: Không thấy cột label `{label_col}` trong CSV.")

    # Chuẩn hóa key về string để merge
    for col in key_cols:
        image_df[col] = image_df[col].astype(str).str.strip()
        label_df[col] = label_df[col].astype(str).str.strip()

    # Merge: CSV không có Camera, nên không merge theo Camera
    manifest_df = image_df.merge(
        label_df,
        on=key_cols,
        how="left",
        indicator=True
    )

    missing_labels_df = manifest_df[manifest_df["_merge"] == "left_only"]

    labels_without_images_df = manifest_df[manifest_df["_merge"] == "right_only"]

    # Kiểm tra mỗi timestamp có bao nhiêu camera
    camera_count_df = (
        image_df
        .groupby(["Subject", "Activity", "Trial", "Timestamp"])["Camera"]
        .nunique()
        .reset_index(name="num_cameras")
    )

    timestamp_missing_camera_df = camera_count_df[
        camera_count_df["num_cameras"] < 2
    ]

    print("========== FILE EXTENSIONS ==========")
    print(suffix_counter)

    print("\n========== IMAGE SCAN ==========")
    print("Total files seen:", total_files_seen)
    print("Total image files seen:", total_image_files_seen)
    print("Valid timestamp image rows:", len(image_df))

    print("\n========== SKIPPED ==========")
    print("Skipped bad structure:", len(skipped_bad_structure))
    print("Skipped bad folder number:", len(skipped_bad_number))
    print("Skipped non-original timestamp filename:", len(skipped_non_timestamp_name))

    # print("\n========== DUPLICATE CHECK ==========")
    # print("Duplicate images same camera:", len(duplicate_image_same_camera))
    # print("Duplicate label keys in CSV:", len(duplicate_label_keys))

    print("\n========== MERGE STATUS ==========")
    print(manifest_df["_merge"].value_counts())

    if label_col in manifest_df.columns:
        print("\n========== LABEL STATUS ==========")
        print("Images with label:", manifest_df[label_col].notna().sum())
        print("Images without label:", manifest_df[label_col].isna().sum())

    print("\n========== CAMERA COUNT PER TIMESTAMP ==========")
    print(camera_count_df["num_cameras"].value_counts().sort_index())
    print("Timestamp missing camera:", len(timestamp_missing_camera_df))

    print("\n========== NA COUNT BY COLUMN ==========")
    print(manifest_df.isna().sum())

    # Lưu debug files
    pd.DataFrame(skipped_bad_structure).to_csv(
        "skipped_bad_structure.csv",
        index=False
    )

    pd.DataFrame(skipped_bad_number).to_csv(
        "skipped_bad_number.csv",
        index=False
    )

    pd.DataFrame(skipped_non_timestamp_name).to_csv(
        "skipped_non_timestamp_name.csv",
        index=False
    )

    missing_labels_df.to_csv(
        "missing_labels.csv",
        index=False
    )

    camera_count_df.to_csv(
        "camera_count_per_timestamp.csv",
        index=False
    )

    labels_without_images_df.to_csv(
        'sub789_labels_without_images_df.csv',
        index=False
    )
    timestamp_missing_camera_df.to_csv(
        "timestamp_missing_camera.csv",
        index=False
    )

    # Lưu manifest cuối cùng
    manifest_df.to_csv(output_csv_path, index=False)

    return manifest_df, image_df, label_df

In [2]:
manifest_df, image_df, label_df = build_frame_manifest(
    image_root="/kaggle/input/datasets/kitvanh/upfall789cam12",
    label_csv_path="/kaggle/input/datasets/kitvanh/sub789-labels/Labels.csv",
    output_csv_path="/kaggle/working/frame_manifest.csv",
    label_col="Label"
)

========== FILE EXTENSIONS ==========
Counter({'.png': 100200, '.json': 20})

========== IMAGE SCAN ==========
Total files seen: 100220
Total image files seen: 100200
Valid timestamp image rows: 100200

========== SKIPPED ==========
Skipped bad structure: 0
Skipped bad folder number: 0
Skipped non-original timestamp filename: 0

========== MERGE STATUS ==========
_merge
both          100199
left_only          1
right_only         0
Name: count, dtype: int64

========== LABEL STATUS ==========
Images with label: 100199
Images without label: 1

========== CAMERA COUNT PER TIMESTAMP ==========
num_cameras
1     8683
2    40702
Name: count, dtype: int64
Timestamp missing camera: 8683

========== NA COUNT BY COLUMN ==========
image_path    0
Subject       0
Activity      0
Trial         0
Camera        0
Timestamp     0
Row_Index     1
Tag           1
Label         1
_merge        0
dtype: int64


In [3]:
manifest_df['Label'] = manifest_df['Label'].fillna(0)

In [4]:
manifest_df.notna().sum()

image_path    100200
Subject       100200
Activity      100200
Trial         100200
Camera        100200
Timestamp     100200
Row_Index     100199
Tag           100199
Label         100200
_merge        100200
dtype: int64

In [5]:
manifest_df[manifest_df['Row_Index'].isna()]

,image_path,Subject,Activity,Trial,Camera,Timestamp,Row_Index,Tag,Label,_merge
54420,Subject7/Activity1/Trial1/Camera2/2018-07-06T1...,7,1,1,2,2018-07-06T12:42:47.2340654,NaN,NaN,0.0,left_only


In [6]:
manifest_df_post = manifest_df.drop(columns=['_merge','Row_Index'])
manifest_df_post

,image_path,Subject,Activity,Trial,Camera,Timestamp,Tag,Label
0,Subject9/Activity4/Trial3/Camera1/2018-07-09T1...,9,4,3,1,2018-07-09T12:06:28.223986,4.0,4.0
1,Subject9/Activity4/Trial3/Camera1/2018-07-09T1...,9,4,3,1,2018-07-09T12:06:30.740560,11.0,11.0
2,Subject9/Activity4/Trial3/Camera1/2018-07-09T1...,9,4,3,1,2018-07-09T12:06:32.491887,11.0,11.0
3,Subject9/Activity4/Trial3/Camera1/2018-07-09T1...,9,4,3,1,2018-07-09T12:06:30.269637,11.0,11.0
4,Subject9/Activity4/Trial3/Camera1/2018-07-09T1...,9,4,3,1,2018-07-09T12:06:35.445781,11.0,11.0
...,...,...,...,...,...,...,...,...
100195,Subject8/Activity7/Trial1/Camera2/2018-07-09T1...,8,7,1,2,2018-07-09T10:30:13.970569,7.0,7.0
100196,Subject8/Activity7/Trial1/Camera2/2018-07-09T1...,8,7,1,2,2018-07-09T10:29:56.336005,7.0,7.0
100197,Subject8/Activity7/Trial1/Camera2/2018-07-09T1...,8,7,1,2,2018-07-09T10:29:54.961341,7.0,7.0
100198,Subject8/Activity7/Trial1/Camera2/2018-07-09T1...,8,7,1,2,2018-07-09T10:30:14.297454,7.0,7.0


In [7]:
manifest_df_post.notna().sum()

image_path    100200
Subject       100200
Activity      100200
Trial         100200
Camera        100200
Timestamp     100200
Tag           100199
Label         100200
dtype: int64

In [8]:
manifest_df_post.to_csv("SUB789_MANIFEST.csv")

# SUB12-17

In [9]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter


def normalize_image_timestamp_strict(filename_stem):
    """
    Nhận timestamp ảnh dạng:
    YYYY-MM-DDTHH_MM_SS
    hoặc
    YYYY-MM-DDTHH_MM_SS.microsecond

    Nếu phía sau có _copy, _1, _2... thì tự cắt bỏ.
    """

    text = str(filename_stem).strip()

    pattern = r"^(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2}(?:\.\d+)?)(?:[\s_-].*)?$"

    match = re.match(pattern, text)

    if not match:
        print(filename_stem)
        return None

    text = match.group(1)

    date_part, time_part = text.split("T", 1)

    time_part = time_part.replace("_", ":", 2)

    return date_part + "T" + time_part


def parse_dataset2_folder_name(folder_name):
    """
    Hỗ trợ cả 2 dạng:
    Subject12_Activity10_Trial1_Camera1
    Subject13Activity10Trial1Camera1

    Trả về:
    subject, activity, trial, camera
    """

    pattern = r"^Subject(\d+)_?Activity(\d+)_?Trial(\d+)_?Camera(\d+)$"

    match = re.match(pattern, folder_name)

    if not match:
        print(folder_name)
        return None

    subject, activity, trial, camera = match.groups()

    return subject, activity, trial, camera


def build_frame_manifest_dataset2(
    image_root,
    label_csv_path,
    output_csv_path,
    label_col="Label"
):
    image_root = Path(image_root)

    rows = []

    skipped_bad_folder_name = []
    skipped_non_timestamp_name = []
    skipped_non_image_files = []

    suffix_counter = Counter()
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

    total_files_seen = 0
    total_image_files_seen = 0
    total_valid_image_rows = 0

    # Tìm tất cả folder có dạng Subject12_Activity10_Trial1_Camera1
    for folder_path in image_root.rglob("*"):

        if not folder_path.is_dir():
            continue

        parsed = parse_dataset2_folder_name(folder_path.name)
        if parsed is None:
            continue

        subject, activity, trial, camera = parsed

        # Iterate ảnh nằm trực tiếp bên trong folder này
        for img_path in folder_path.iterdir():

            if not img_path.is_file():
                continue

            total_files_seen += 1
            suffix_counter[img_path.suffix.lower()] += 1

            if img_path.suffix.lower() not in image_extensions:
                skipped_non_image_files.append({
                    "file_path": str(img_path),
                    "suffix": img_path.suffix.lower()
                })
                continue

            total_image_files_seen += 1

            timestamp = normalize_image_timestamp_strict(img_path.stem)

            if timestamp is None:
                skipped_non_timestamp_name.append({
                    "image_path": str(img_path),
                    "relative_path": str(img_path.relative_to(image_root)),
                    "filename_stem": img_path.stem,
                    "reason": "Tên file không phải timestamp hợp lệ hoặc format khác"
                })
                continue

            rows.append({
                "image_path": str(img_path.relative_to(image_root)),
                "Subject": str(subject),
                "Activity": str(activity),
                "Trial": str(trial),
                "Camera": str(camera),
                "Timestamp": timestamp
            })

            total_valid_image_rows += 1

    image_df = pd.DataFrame(rows)

    label_df = pd.read_csv(label_csv_path)

    key_cols = ["Subject", "Activity", "Trial", "Timestamp"]

    missing_cols = [col for col in key_cols if col not in label_df.columns]
    if missing_cols:
        raise ValueError(f"CSV thiếu các cột bắt buộc: {missing_cols}")

    if label_col not in label_df.columns:
        print(f"Cảnh báo: Không thấy cột label `{label_col}` trong CSV.")

    # Nếu không tìm thấy ảnh hợp lệ thì báo rõ
    if image_df.empty:
        raise ValueError(
            "Không tìm thấy ảnh hợp lệ nào. "
            "Hãy kiểm tra lại folder có dạng Subject12_Activity10_Trial1_Camera1 "
            "và tên ảnh có đúng dạng timestamp không."
        )

    # Chuẩn hóa key về string để merge
    for col in key_cols:
        image_df[col] = image_df[col].astype(str).str.strip()
        label_df[col] = label_df[col].astype(str).str.strip()

    # Merge label theo Timestamp + Subject + Activity + Trial
    # Không merge theo Camera vì label CSV không có Camera
    manifest_df = image_df.merge(
        label_df,
        on=key_cols,
        how="left",
        indicator=True
    )

    missing_labels_df = manifest_df[manifest_df["_merge"] == "left_only"]

    labels_without_images_df = manifest_df[manifest_df["_merge"] == "right_only"]

    # Kiểm tra mỗi timestamp có bao nhiêu camera
    camera_count_df = (
        image_df
        .groupby(["Subject", "Activity", "Trial", "Timestamp"])["Camera"]
        .nunique()
        .reset_index(name="num_cameras")
    )

    timestamp_missing_camera_df = camera_count_df[
        camera_count_df["num_cameras"] < 2
    ]

    print("========== FILE EXTENSIONS ==========")
    print(suffix_counter)

    print("\n========== IMAGE SCAN ==========")
    print("Total files seen inside matched folders:", total_files_seen)
    print("Total image files seen:", total_image_files_seen)
    print("Valid timestamp image rows:", len(image_df))

    print("\n========== SKIPPED ==========")
    print("Skipped non-image files:", len(skipped_non_image_files))
    print("Skipped non-original timestamp filename:", len(skipped_non_timestamp_name))

    print("\n========== MERGE STATUS ==========")
    print(manifest_df["_merge"].value_counts())

    if label_col in manifest_df.columns:
        print("\n========== LABEL STATUS ==========")
        print("Images with label:", manifest_df[label_col].notna().sum())
        print("Images without label:", manifest_df[label_col].isna().sum())

    print("\n========== CAMERA COUNT PER TIMESTAMP ==========")
    print(camera_count_df["num_cameras"].value_counts().sort_index())
    print("Timestamp missing camera:", len(timestamp_missing_camera_df))

    print("\n========== NA COUNT BY COLUMN ==========")
    print(manifest_df.isna().sum())

    # Lưu debug files
    pd.DataFrame(skipped_non_image_files).to_csv(
        "skipped_non_image_files.csv",
        index=False
    )

    pd.DataFrame(skipped_non_timestamp_name).to_csv(
        "skipped_non_timestamp_name.csv",
        index=False
    )

    missing_labels_df.to_csv(
        "missing_labels.csv",
        index=False
    )

    camera_count_df.to_csv(
        "camera_count_per_timestamp.csv",
        index=False
    )

    timestamp_missing_camera_df.to_csv(
        "timestamp_missing_camera.csv",
        index=False
    )

    labels_without_images_df.to_csv(
        "labels_without_images_df.csv",
        index=False
    )
    # Lưu manifest cuối cùng
    manifest_df.to_csv(output_csv_path, index=False)

    return manifest_df, image_df, label_df

In [12]:
manifest_df, image_df, label_df = build_frame_manifest_dataset2(
    image_root="/kaggle/input/datasets/nguyenhongphat112/upfall-subject13",
    label_csv_path="/kaggle/input/datasets/kitvanh/sub789-labels/Labels.csv",
    output_csv_path="SUB12_17_MANIFEST.csv",
    label_col="Label"
)

Camera_Subject17
Camera_Subject14
Camera_Subject15
Camera_Subject12
Camera_Subject16
Camera_Subject11
Camera_Subject13
Camera_Subject17
Camera_Subject14
Camera_Subject15
Camera_Subject12
Camera_Subject16
Camera_Subject11
Camera_Subject13
========== FILE EXTENSIONS ==========
Counter({'.png': 244732})

========== IMAGE SCAN ==========
Total files seen inside matched folders: 244732
Total image files seen: 244732
Valid timestamp image rows: 244732

========== SKIPPED ==========
Skipped non-image files: 0
Skipped non-original timestamp filename: 0

========== MERGE STATUS ==========
_merge
both          244732
left_only          0
right_only         0
Name: count, dtype: int64

========== LABEL STATUS ==========
Images with label: 244732
Images without label: 0

========== CAMERA COUNT PER TIMESTAMP ==========
num_cameras
2    122366
Name: count, dtype: int64
Timestamp missing camera: 0

========== NA COUNT BY COLUMN ==========
image_path    0
Subject       0
Activity      0
Trial        

In [13]:
manifest_df['Camera'].value_counts()

Camera
2    122366
1    122366
Name: count, dtype: int64

In [14]:
manifest_df.drop(columns=['Row_Index','_merge','Row_Index']).to_csv('SUB11-17_MANIFEST.csv')

In [15]:
manifest_df

,image_path,Subject,Activity,Trial,Camera,Timestamp,Row_Index,Tag,Label,_merge
0,Camera_Subject17/Camera_Subject17/Subject17_Ac...,17,1,1,2,2018-07-12T11:54:28.884751,277416,11,11,both
1,Camera_Subject17/Camera_Subject17/Subject17_Ac...,17,1,1,2,2018-07-12T11:54:23.242681,277326,11,11,both
2,Camera_Subject17/Camera_Subject17/Subject17_Ac...,17,1,1,2,2018-07-12T11:54:28.537827,277410,11,11,both
3,Camera_Subject17/Camera_Subject17/Subject17_Ac...,17,1,1,2,2018-07-12T11:54:26.288321,277373,11,11,both
4,Camera_Subject17/Camera_Subject17/Subject17_Ac...,17,1,1,2,2018-07-12T11:54:26.865360,277382,11,11,both
...,...,...,...,...,...,...,...,...,...,...
244727,Camera_Subject13/Camera_Subject13/Subject13Act...,13,11,3,1,2018-07-11T13:06:12.865559,226146,11,11,both
244728,Camera_Subject13/Camera_Subject13/Subject13Act...,13,11,3,1,2018-07-11T13:06:33.635810,226494,11,11,both
244729,Camera_Subject13/Camera_Subject13/Subject13Act...,13,11,3,1,2018-07-11T13:06:12.273475,226135,11,11,both
244730,Camera_Subject13/Camera_Subject13/Subject13Act...,13,11,3,1,2018-07-11T13:06:13.886807,226162,11,11,both


# Fusion Between 2 Manifest

In [17]:
import pandas as pd
sub789 = pd.read_csv('/kaggle/working/SUB789_MANIFEST.csv')
sub11_17 = pd.read_csv('/kaggle/working/SUB11-17_MANIFEST.csv')

sub789_11_17 = pd.concat([sub789, sub11_17], ignore_index=True)

In [18]:
len(sub789) + len(sub11_17)

344932

In [19]:
sub789_11_17.to_csv('SUB789-11_17_MANIFEST.csv')